In [ ]:
'Cellule 1'
from google.colab import drive

drive.mount('/content/drive')

import subprocess
import sys

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    'numpy',
    'pandas',
    'matplotlib',
    'scikit-learn',
    'torch',
])


In [ ]:
'Cellule 2'
from pathlib import Path
from types import SimpleNamespace
import sys

CFG = SimpleNamespace(
    project_dir=Path('/content/drive/MyDrive/New project/python'),
    data_root=Path('/content/drive/MyDrive/projoRN'),
    processed_env_dir=Path('/content/drive/MyDrive/New project/python/processed_env_real_csv'),
    output_dir=Path('/content/drive/MyDrive/New project/python/part3_outputs_improved'),
    states=['arkansas', 'california'],
    configs=['baseline', 'climate', 'soil', 'topography', 'all'],
    epochs=250,
    batch_size=32,
    lr=1e-3,
    learning_rate=1e-3,
    weight_decay=1e-4,
    dropout=0.1,
    patience=30,
    early_stopping_patience=30,
    seed=2021,
    use_env_covariates=True,
    use_weighted_sampler=True,
    class_weight_power=0.5,
    label_smoothing=0.05,
    gradient_clip_norm=1.0,
    static_hidden_dim=32,
    cpu=False,
)

CFG.output_dir.mkdir(parents=True, exist_ok=True)

if not CFG.project_dir.exists():
    raise FileNotFoundError(f'project_dir introuvable: {CFG.project_dir}')

if not CFG.processed_env_dir.exists():
    raise FileNotFoundError(f'processed_env_dir introuvable: {CFG.processed_env_dir}')

if str(CFG.project_dir) not in sys.path:
    sys.path.insert(0, str(CFG.project_dir))


def resolve_bundle_paths(processed_env_dir: Path, state: str) -> tuple[Path, Path]:
    candidates = [
        (
            processed_env_dir / f'{state}_mctnet_env_dataset.npz',
            processed_env_dir / f'{state}_mctnet_env_dataset.json',
        ),
        (
            processed_env_dir / f'{state}_mctnet_env_dataset_FULL_TEMPORAL.npz',
            processed_env_dir / f'{state}_mctnet_env_dataset_FULL_TEMPORAL.json',
        ),
        (
            processed_env_dir / f'{state}_mctnet_dataset.npz',
            processed_env_dir / f'{state}_mctnet_dataset.json',
        ),
    ]
    for npz_path, json_path in candidates:
        if npz_path.exists() and json_path.exists():
            return npz_path, json_path
    tried = '\n'.join(
        [f'  - {npz_candidate.name} | {json_candidate.name}' for npz_candidate, json_candidate in candidates]
    )
    message = f'Aucun bundle trouve pour {state} dans {processed_env_dir}.\n{tried}'
    raise FileNotFoundError(message)


print('project_dir:', CFG.project_dir)
print('data_root:', CFG.data_root)
print('processed_env_dir:', CFG.processed_env_dir)
print('output_dir:', CFG.output_dir)
for state in CFG.states:
    try:
        npz_path, json_path = resolve_bundle_paths(CFG.processed_env_dir, state)
        print(f'[{state}] bundle:', npz_path.name, '| metadata:', json_path.name)
    except FileNotFoundError as error:
        print(f'[{state}]', error)


In [ ]:
'Cellule 3'
import json
import numpy as np

from mctnet_part3_improved import (
    MCTNetPart3Improved,
    compute_vi_timeseries_improved,
    build_class_weights,
    normalize_vi_columns_improved,
)
from run_part3_improved_experiment import CropDatasetImproved, evaluate_improved

for state in CFG.states:
    npz_path, json_path = resolve_bundle_paths(CFG.processed_env_dir, state)

    print(f'\n[{state}]')
    print('npz:', npz_path)
    print('json:', json_path)

    bundle = np.load(npz_path)
    metadata = json.loads(json_path.read_text(encoding='utf-8'))

    x_train = bundle['x_train']
    valid_mask_train = bundle['valid_mask_train']
    y_train = bundle['y_train']
    x_train_vi = compute_vi_timeseries_improved(x_train)

    print('  x_train:', x_train.shape)
    print('  valid_mask_train:', valid_mask_train.shape)
    print('  y_train:', y_train.shape)
    print('  x_train_vi:', x_train_vi.shape)
    print('  num_classes:', metadata.get('num_classes'))
    print('  class_name_to_index:', metadata.get('class_name_to_index', {}))
    print('  ablation_configs:', sorted(metadata.get('ablation_configs', {}).keys()))

    dynamic_env_train = None
    static_env_train = None

    if 'dynamic_env_train' in bundle:
        dynamic_env_train = bundle['dynamic_env_train']
        print('  dynamic_env_train:', dynamic_env_train.shape)
    if 'static_env_train' in bundle:
        static_env_train = bundle['static_env_train']
        print('  static_env_train:', static_env_train.shape)
    if 'env_train' in bundle:
        env_train = bundle['env_train']
        print('  env_train:', env_train.shape)

    baseline_ds = CropDatasetImproved(
        x=x_train_vi[:4],
        mask=valid_mask_train[:4],
        y=y_train[:4],
        dynamic_env=None,
        static_env=None,
    )
    x_base, mask_base, y_base, dynamic_base, static_base = baseline_ds[0]
    print('  baseline sample:', tuple(x_base.shape), tuple(mask_base.shape), int(y_base), tuple(dynamic_base.shape), tuple(static_base.shape))

    if dynamic_env_train is not None or static_env_train is not None:
        improved_ds = CropDatasetImproved(
            x=x_train_vi[:4],
            mask=valid_mask_train[:4],
            y=y_train[:4],
            dynamic_env=None if dynamic_env_train is None else dynamic_env_train[:4],
            static_env=None if static_env_train is None else static_env_train[:4],
        )
        x_imp, mask_imp, y_imp, dynamic_imp, static_imp = improved_ds[0]
        print('  improved sample:', tuple(x_imp.shape), tuple(mask_imp.shape), int(y_imp), tuple(dynamic_imp.shape), tuple(static_imp.shape))
        print('  temporal_input_dim baseline:', x_imp.shape[-1])


In [ ]:
'Cellule 4'
from run_part3_improved_experiment import run_part3_full_ablation_improved, set_seed

set_seed(CFG.seed)
results = run_part3_full_ablation_improved(
    processed_env_dir=CFG.processed_env_dir,
    output_dir=CFG.output_dir,
    states=CFG.states,
    args=CFG,
)

print(results)


In [ ]:
'Cellule 5'
from IPython.display import Image as IPImage, display

for state in CFG.states:
    for config in CFG.configs:
        png = CFG.output_dir / state / f'learning_curves_improved_{config}.png'
        if png.exists():
            print(state, config, png.name)
            display(IPImage(filename=str(png)))


In [ ]:
'Cellule 6'
from IPython.display import Image as IPImage, display

for state in CFG.states:
    for config in CFG.configs:
        png = CFG.output_dir / state / f'confusion_matrix_improved_{config}.png'
        if png.exists():
            print(state, config, png.name)
            display(IPImage(filename=str(png)))


In [ ]:
'Cellule 7'
import pandas as pd

summary_path = CFG.output_dir / 'part3_improved_ablation_summary.csv'
mctnet_baseline = {
    'arkansas': {'OA': 0.968, 'Kappa': 0.951, 'F1': 0.933},
    'california': {'OA': 0.852, 'Kappa': 0.806, 'F1': 0.829},
}

if not summary_path.exists():
    print(f'Fichier de synthese introuvable: {summary_path}')
    print('Lance d abord la cellule 4 pour generer les resultats Part 3 improved.')
else:
    summary = pd.read_csv(summary_path)
    if summary.empty:
        print(f'Le fichier {summary_path.name} est vide.')
    else:
        summary['state'] = pd.Categorical(summary['state'], categories=CFG.states, ordered=True)
        summary['config'] = pd.Categorical(summary['config'], categories=CFG.configs, ordered=True)
        summary = summary.sort_values(['state', 'config']).reset_index(drop=True)

        expected = pd.MultiIndex.from_product([CFG.states, CFG.configs], names=['state', 'config']).to_frame(index=False)
        progress = expected.merge(summary[['state', 'config']], on=['state', 'config'], how='left', indicator=True)
        progress['status'] = progress['_merge'].map({'both': 'done', 'left_only': 'pending'})
        progress = progress[['state', 'config', 'status']]

        print('Resultats disponibles')
        display(summary)

        print('Progression des experiences')
        display(progress)

        oa_pivot = summary.pivot_table(index='config', columns='state', values='OA', aggfunc='first').reindex(index=CFG.configs, columns=CFG.states)
        kappa_pivot = summary.pivot_table(index='config', columns='state', values='Kappa', aggfunc='first').reindex(index=CFG.configs, columns=CFG.states)
        f1_pivot = summary.pivot_table(index='config', columns='state', values='F1_macro', aggfunc='first').reindex(index=CFG.configs, columns=CFG.states)

        print('OA pivot')
        display(oa_pivot)
        print('Kappa pivot')
        display(kappa_pivot)
        print('F1 pivot')
        display(f1_pivot)

        improvement_rows = []
        for _, row in summary.iterrows():
            state = str(row['state'])
            if state not in mctnet_baseline:
                continue
            base = mctnet_baseline[state]
            improvement_rows.append({
                'state': state,
                'config': str(row['config']),
                'delta_OA': float(row['OA']) - float(base['OA']),
                'delta_Kappa': float(row['Kappa']) - float(base['Kappa']),
                'delta_F1': float(row['F1_macro']) - float(base['F1']),
            })

        improvements = pd.DataFrame(improvement_rows)
        if improvements.empty:
            print('Aucune amelioration a comparer pour le moment.')
        else:
            improvements['state'] = pd.Categorical(improvements['state'], categories=CFG.states, ordered=True)
            improvements['config'] = pd.Categorical(improvements['config'], categories=CFG.configs, ordered=True)
            improvements = improvements.sort_values(['state', 'config']).reset_index(drop=True)
            print('Improvements vs MCTNet original')
            display(improvements)

        best_by_state = summary.sort_values(['state', 'F1_macro'], ascending=[True, False]).groupby('state', as_index=False).head(1)
        print('Meilleure configuration actuellement disponible par etat (selon F1_macro)')
        display(best_by_state[['state', 'config', 'OA', 'Kappa', 'F1_macro', 'best_epoch', 'temporal_input_dim', 'static_input_dim']])


In [ ]:
'Cellule 8'
from IPython.display import Image as IPImage, display
from run_part3_improved_experiment import plot_ablation_comparison_improved

summary_path = CFG.output_dir / 'part3_improved_ablation_summary.csv'
barplot_path = CFG.output_dir / 'part3_improved_comparison_barplot.png'

if not summary_path.exists():
    print(f'Fichier de synthese introuvable: {summary_path}')
    print('Lance d abord la cellule 4 pour generer les resultats Part 3 improved.')
else:
    plot_ablation_comparison_improved(
        summary_csv=summary_path,
        output_dir=CFG.output_dir,
    )
    if barplot_path.exists():
        display(IPImage(filename=str(barplot_path)))
    else:
        print(f'Barplot non genere: {barplot_path}')
